# 02 — Data Cleaning

Minimal, reproducible cleaning of the NovoStar datasets.

Decisions:
- Remove exact duplicate customer and order records.
- Preserve order-item rows.
- Flag suspicious order dates, negative totals, and invalid campaign timing.
- Standardize customer country values.
- Keep returned and cancelled orders with their original status.
- Preserve raw datasets and export cleaned tables to `data/processed/`.

## Loading and dropping dupes

In [8]:
import pandas as pd
from IPython.display import display

from src.data_loader import load_data
from src.config import PROCESSED_DATA_DIR

data = load_data()

customers = data["customers"].drop_duplicates().copy()
products = data["products"].copy()
orders = data["orders"].drop_duplicates().copy()
order_items = data["order_items"].copy()
campaigns = data["campaigns"].copy()
support = data["support"].copy()

## Standardizing country names

In [9]:
country_mapping = {
    "Deutschland": "Germany",
    "DE": "Germany",
    "GER": "Germany",
    "germany": "Germany",
    "Österreich": "Austria",
    "AT": "Austria",
    "austria": "Austria",
    "Holland": "Netherlands",
    "NL": "Netherlands",
    "FR": "France",
    "PL": "Poland",
    "CH": "Switzerland",
    "Schweiz": "Switzerland",
}

customers["country"] = (
    customers["country"]
    .replace(country_mapping)
)

## Flagging data

In [10]:
# Suspicious or placeholder dates
orders["flag_placeholder_date"] = (
    orders["order_date"] == pd.Timestamp("2019-01-01")
)

# Orders before customer signup
orders = orders.merge(
    customers[["customer_id", "signup_date"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

orders["flag_before_signup"] = (
    orders["order_date"] < orders["signup_date"]
)

orders = orders.drop(columns="signup_date")

# Negative recorded order totals
orders["flag_negative_total"] = (
    orders["total_amount"] < 0
)

# Orders attributed outside the campaign period
campaign_dates = campaigns[
    ["campaign_id", "start_date", "end_date"]
]

orders = orders.merge(
    campaign_dates,
    on="campaign_id",
    how="left",
    validate="many_to_one",
)

orders["flag_outside_campaign"] = (
    orders["campaign_id"].notna()
    & (
        (orders["order_date"] < orders["start_date"])
        | (orders["order_date"] > orders["end_date"])
    )
)

orders = orders.drop(columns=["start_date", "end_date"])

In [11]:
period_start = pd.Timestamp("2024-07-01")
period_end = pd.Timestamp("2026-06-30")

orders["in_reporting_period"] = (
    orders["order_date"].between(period_start, period_end)
    & ~orders["flag_placeholder_date"]
)

orders["is_completed"] = (
    orders["status"].eq("completed")
)

orders["is_revenue_eligible"] = (
    orders["in_reporting_period"]
    & orders["is_completed"]
    & ~orders["flag_negative_total"]
)

## Pushing data to processed

In [12]:
processed = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "campaigns": campaigns,
    "support_tickets": support,
}

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

for name, df in processed.items():
    df.to_csv(
        PROCESSED_DATA_DIR / f"{name}.csv",
        index=False,
    )

summary = pd.DataFrame({
    "dataset": list(processed.keys()),
    "rows": [len(df) for df in processed.values()],
})

display(summary)

print("\nOrder flags:")
display(
    orders[
        [
            "flag_placeholder_date",
            "flag_before_signup",
            "flag_negative_total",
            "flag_outside_campaign",
            "in_reporting_period",
            "is_revenue_eligible",
        ]
    ].sum().to_frame("count")
)

print("\nProcessed datasets saved.")

,dataset,rows
0,customers,3120
1,products,150
2,orders,14940
3,order_items,29887
4,campaigns,12
5,support_tickets,1540



Order flags:


,count
flag_placeholder_date,30
flag_before_signup,179
flag_negative_total,45
flag_outside_campaign,10
in_reporting_period,14910
is_revenue_eligible,12811



Processed datasets saved.
